In [1]:
import torch
import torch.nn as nn
import os
import numpy as np
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import random
import torch.optim as optim
import dotenv
print("Done")

Done


In [2]:
dotenv.load_dotenv()

path = os.getenv("MAIN_PATH")

In [3]:
test_path = os.path.join(path,"Test","Test")
train_path = os.path.join(path,"Train","Train")
op = "output"
#data imbalance check
for i in os.listdir(train_path):
    print(len(os.listdir(os.path.join(train_path,i))))



3066
2975
2979
2962
2926
3133
2921
2711
3371
2959


In [10]:
#making train path
os.makedirs(os.path.join(op,"Train"),exist_ok=True)
os.makedirs(os.path.join(op,"Test"),exist_ok=True)

In [14]:
import random

# Define 'op' as it's used in this context for the output directory
train_output_path = os.path.join(path, "Train","Train")



print("Checking for processed training data to randomly remove half...")

if os.path.exists(train_output_path):
    for class_name in os.listdir(train_output_path):
        class_dir = os.path.join(train_output_path, class_name)
        if os.path.isdir(class_dir):
            all_pt_files = [f for f in os.listdir(class_dir) if f.endswith('.bin')]
            random.shuffle(all_pt_files)

            num_files_to_remove = len(all_pt_files) // 2
            files_to_remove = all_pt_files[:num_files_to_remove]

            for filename in files_to_remove:
                file_path_to_delete = os.path.join(class_dir, filename)
                os.remove(file_path_to_delete)
            print(f"Removed {num_files_to_remove} files from class '{class_name}' in training data.")
else:
    print(f"Training output directory '{train_output_path}' does not exist yet. No processed files to remove.")

Checking for processed training data to randomly remove half...
Removed 3065 files from class '3' in training data.
Removed 2974 files from class '9' in training data.
Removed 2979 files from class '2' in training data.
Removed 2961 files from class '0' in training data.
Removed 2925 files from class '8' in training data.
Removed 3132 files from class '7' in training data.
Removed 2921 files from class '4' in training data.
Removed 2710 files from class '5' in training data.
Removed 3371 files from class '1' in training data.
Removed 2959 files from class '6' in training data.


In [15]:
def process_data(path,sub_path):
    for i in os.listdir(path):
      cat_path = os.path.join(path,i)
      os.makedirs(os.path.join(op, sub_path, i),exist_ok=True)

      for j in (os.listdir(cat_path)):
        check_size = np.fromfile(os.path.join(cat_path,j),dtype = np.uint8)
        check_size = check_size.reshape(-1,5)
        # print(check_size.shape)
        X = check_size[:,0]
        Y = check_size[:,1]
        polarity =(check_size[:,2] & 0x80 ) >> 7
        timestamps = (check_size[:, 3].astype(np.uint32) << 8) | check_size[:, 4]
        grid = np.zeros((30, 2, 34, 34), dtype=np.float32)
        for x, y, pol, t in zip(X, Y, polarity, timestamps):
            time_bin = int(t // 10000)

            if time_bin < 30 and x < 34 and y < 34:
                grid[time_bin, pol, y, x] = 1.0
        grid = grid.transpose(1, 0, 2, 3)
        torch_set = torch.from_numpy(grid)

        pt_filename = j.replace(".bin", ".pt")
        save_sub_path = os.path.join(op, sub_path, i, pt_filename)

        torch.save(torch_set, save_sub_path)
        print(f"Saved file {pt_filename} with shape {torch_set.shape}")

In [16]:
process_data(train_path,"Train")

Saved file 41405.pt with shape torch.Size([2, 30, 34, 34])
Saved file 15252.pt with shape torch.Size([2, 30, 34, 34])
Saved file 29671.pt with shape torch.Size([2, 30, 34, 34])
Saved file 27841.pt with shape torch.Size([2, 30, 34, 34])
Saved file 56054.pt with shape torch.Size([2, 30, 34, 34])
Saved file 41226.pt with shape torch.Size([2, 30, 34, 34])
Saved file 01379.pt with shape torch.Size([2, 30, 34, 34])
Saved file 49571.pt with shape torch.Size([2, 30, 34, 34])
Saved file 45276.pt with shape torch.Size([2, 30, 34, 34])
Saved file 49411.pt with shape torch.Size([2, 30, 34, 34])
Saved file 42678.pt with shape torch.Size([2, 30, 34, 34])
Saved file 47873.pt with shape torch.Size([2, 30, 34, 34])
Saved file 41516.pt with shape torch.Size([2, 30, 34, 34])
Saved file 20729.pt with shape torch.Size([2, 30, 34, 34])
Saved file 04236.pt with shape torch.Size([2, 30, 34, 34])
Saved file 22239.pt with shape torch.Size([2, 30, 34, 34])
Saved file 09731.pt with shape torch.Size([2, 30, 34, 34

In [19]:
for i in os.listdir(test_path):
    print(len(os.listdir(os.path.join(test_path,i))))


1010
1009
1032
980
974
1028
982
892
1135
958


In [15]:
import shutil
import random
import numpy as np


# Clean up existing output directories to ensure a fresh split
shutil.rmtree(os.path.join(op, "Test"), ignore_errors=True)
shutil.rmtree(os.path.join(op, "Validate"), ignore_errors=True)

# Define raw test path (assuming 'path' variable is still in scope from previous cells)
test_path_raw = test_path

# Create output directories
os.makedirs(os.path.join(op, "Test"), exist_ok=True)
os.makedirs(os.path.join(op, "Validate"), exist_ok=True)

for class_name in os.listdir(test_path_raw):
    class_raw_path = os.path.join(test_path_raw, class_name)

    # Ensure output directories exist for this class
    os.makedirs(os.path.join(op, "Test", class_name), exist_ok=True)
    os.makedirs(os.path.join(op, "Validate", class_name), exist_ok=True)

    all_files_in_class = [f for f in os.listdir(class_raw_path) if f.endswith('.bin')]
    random.shuffle(all_files_in_class) # Shuffle to ensure random split

    # Split into two halves
    half_point = len(all_files_in_class) // 2
    test_split_files = all_files_in_class[:half_point]
    validate_split_files = all_files_in_class[half_point:]

    print(f"Class {class_name}: {len(test_split_files)} files for Test, {len(validate_split_files)} files for Validate")

    # Process and save files for the 'Test' split
    for filename in test_split_files:
        full_file_path = os.path.join(class_raw_path, filename)
        check_size = np.fromfile(full_file_path, dtype=np.uint8)
        check_size = check_size.reshape(-1, 5)
        X = check_size[:, 0]
        Y = check_size[:, 1]
        polarity = (check_size[:, 2] & 0x80) >> 7
        timestamps = (check_size[:, 3].astype(np.uint32) << 8) | check_size[:, 4]
        grid = np.zeros((30, 2, 34, 34), dtype=np.float32)
        for x, y, pol, t in zip(X, Y, polarity, timestamps):
            time_bin = int(t // 10000)
            if time_bin < 30 and x < 34 and y < 34:
                grid[time_bin, pol, y, x] = 1.0
        grid = grid.transpose(1, 0, 2, 3)
        torch_set = torch.from_numpy(grid)
        pt_filename = filename.replace(".bin", ".pt")

        target_dir = os.path.join(op, "Test", class_name)
        save_path = os.path.join(target_dir, pt_filename)
        torch.save(torch_set, save_path)

    # Process and save files for the 'Validate' split
    for filename in validate_split_files:
        full_file_path = os.path.join(class_raw_path, filename)
        check_size = np.fromfile(full_file_path, dtype=np.uint8)
        check_size = check_size.reshape(-1, 5)
        X = check_size[:, 0]
        Y = check_size[:, 1]
        polarity = (check_size[:, 2] & 0x80) >> 7
        timestamps = (check_size[:, 3].astype(np.uint32) << 8) | check_size[:, 4]
        grid = np.zeros((30, 2, 34, 34), dtype=np.float32)
        for x, y, pol, t in zip(X, Y, polarity, timestamps):
            time_bin = int(t // 10000)
            if time_bin < 30 and x < 34 and y < 34:
                grid[time_bin, pol, y, x] = 1.0
        grid = grid.transpose(1, 0, 2, 3)
        torch_set = torch.from_numpy(grid)
        pt_filename = filename.replace(".bin", ".pt")

        target_dir = os.path.join(op, "Validate", class_name)
        save_path = os.path.join(target_dir, pt_filename)
        torch.save(torch_set, save_path)

Class 3: 505 files for Test, 505 files for Validate
Class 9: 504 files for Test, 505 files for Validate
Class 2: 516 files for Test, 516 files for Validate
Class 0: 490 files for Test, 490 files for Validate
Class 8: 487 files for Test, 487 files for Validate
Class 7: 514 files for Test, 514 files for Validate
Class 4: 491 files for Test, 491 files for Validate
Class 5: 446 files for Test, 446 files for Validate
Class 1: 567 files for Test, 568 files for Validate
Class 6: 479 files for Test, 479 files for Validate


In [16]:
from torchvision.datasets import DatasetFolder

# Define a dummy loader for .pt files
def pt_loader(path):
    return torch.load(path)

# Create DatasetFolder instances
train_dataset = DatasetFolder(
    root=os.path.join(op, "Train"),
    loader=pt_loader,
    extensions=('.pt',)
)

test_dataset = DatasetFolder(
    root=os.path.join(op, "Test"),
    loader=pt_loader,
    extensions=('.pt',)
)

val_dataset = DatasetFolder(
    root=os.path.join(op, "Validate"),
    loader=pt_loader,
    extensions=('.pt',)
)
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
print(f"Number of samples in training : {len(train_dataset)}")
print(f"Number of samples in test : {len(test_dataset)}")
print(f"Number of samples in validation : {len(val_dataset)}")
# print(next(iter(train_loader)))


Number of samples in training : 30003
Number of samples in test : 4999
Number of samples in validation : 5001


In [17]:
class CNN(nn.Module):
    def __init__(self,ip):
        super().__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv3d(in_channels=ip, out_channels=16, kernel_size=3, padding=1),
            nn.BatchNorm3d(16),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=2, stride=2),
            nn.Conv3d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(),
            nn.MaxPool3d(kernel_size=2, stride=2)
        )

        zeros = torch.zeros(1,ip,30,34,34)
        op = self.conv_layer(zeros)
        self.flatten_size = op.shape[1]*op.shape[2]*op.shape[3]* op.shape[4]
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.flatten_size, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_layer(x)
        x = self.classifier(x)
        return x

In [18]:
train_loss = np.array([])
test_loss = np.array([])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ip_channel = 2
model = CNN(ip_channel).to(device)
epochs = 10
learning_rate = 0.0001
loss_func = nn.CrossEntropyLoss()
optimizer = optim.Adam(params=model.parameters(),lr = learning_rate,weight_decay=1e-4)

In [19]:
device

device(type='cuda')

In [ ]:
# Optional: load previously trained weights (prevents evaluating a freshly re-initialized model)
MODEL_PATH = os.path.join(op, "model.pth")
if os.path.isfile(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    print(f"Loaded weights from {MODEL_PATH}")
else:
    print(f"No saved weights found at {MODEL_PATH}. Run the training cell to train the model.")

In [ ]:
for epoch in range(epochs):
    train_loss_perCycle=0
    test_loss_perCycle=0
    model.train()
    for X_train,y_train in train_loader:
        X_train,y_train = X_train.to(device), y_train.long().to(device)
        y_pred = model(X_train)
        loss = loss_func(y_pred, y_train)
        optimizer.zero_grad()
        #Backpropagation
        loss.backward()

        #update
        optimizer.step()
        train_loss_perCycle += loss.item()
    avg_train_loss = train_loss_perCycle/len(train_loader)
    train_loss = np.append(train_loss,avg_train_loss)

    model.eval()
    with torch.no_grad():
        for x_test,y_test in test_loader:
            x_test,y_test = x_test.to(device), y_test.long().to(device)
            y_test_pred = model(x_test)
            loss = loss_func(y_test_pred,y_test)
            test_loss_perCycle += loss.item()
        avg_test_loss = test_loss_perCycle/len(test_loader)
        test_loss = np.append(test_loss,avg_test_loss)
    print(f"Train Loss for round {epoch + 1} is {avg_train_loss} | Test Loss for round {epoch + 1} is {avg_test_loss}")

In [ ]:
model.eval()
pred_batches = []
actual_batches = []
cnt = 0
with torch.inference_mode():
    for X_val, y_val in val_loader:
        X_val, y_val = X_val.to(device), y_val.to(device)
        logits = model(X_val)
        y_val_pred = logits.argmax(dim=1).cpu().numpy()
        print(f"y_val_pred for {cnt} is:", y_val_pred)
        pred_batches.append(y_val_pred)
        actual_batches.append(y_val.cpu().numpy())
        cnt += 1

predicted = np.concatenate(pred_batches, axis=0)
actual = np.concatenate(actual_batches, axis=0)

y_val_pred for 0 is: [6 8 8 6 8 8 6 6 8 8 8 8 6 8 8 8 6 6 6 8 9 8 8 6 5 8 8 8 8 6 8 8 3 6 8 8 5
 8 6 6 9 8 6 8 8 8 8 8 6 8 6 8 8 8 8 8 8 6 9 6 8 8 6 8]
y_val_pred for 1 is: [6 8 8 6 6 8 8 8 9 8 8 8 8 3 9 6 6 8 8 6 8 6 6 8 4 9 8 8 8 6 6 9 6 8 6 8 8
 8 6 6 8 6 6 2 9 8 9 6 9 8 3 8 8 8 4 6 9 6 8 6 8 8 8 3]
y_val_pred for 2 is: [8 6 6 5 8 8 6 6 8 8 8 6 6 6 6 9 8 8 8 8 8 8 6 8 8 8 9 8 8 8 8 8 8 8 8 6 8
 8 8 8 6 6 8 8 6 8 6 8 6 8 8 6 8 6 8 8 6 8 8 6 8 8 8 6]
y_val_pred for 3 is: [6 4 8 6 6 6 8 6 8 6 8 6 8 8 8 6 8 6 6 8 8 8 8 6 6 6 3 8 8 8 6 8 8 8 8 8 8
 8 8 8 6 8 6 6 8 8 8 6 8 9 8 9 8 6 9 8 8 5 8 8 6 3 8 3]
y_val_pred for 4 is: [8 8 6 8 8 6 8 8 8 6 8 8 8 8 8 6 6 8 8 8 8 8 8 8 8 9 5 8 6 5 9 6 8 6 8 6 6
 8 8 3 8 9 8 8 6 6 8 9 8 4 8 6 6 6 8 6 2 8 6 9 6 8 8 8]
y_val_pred for 5 is: [8 9 8 6 8 6 6 8 8 6 8 6 8 8 8 6 8 6 8 8 6 6 8 9 6 6 8 6 8 6 6 6 8 8 6 4 6
 6 8 8 6 8 8 6 6 8 8 8 6 8 8 8 8 6 8 8 3 6 6 6 6 8 6 8]
y_val_pred for 6 is: [3 8 6 8 8 6 8 9 9 8 8 8 8 8 8 8 8 8 6 8 6 8 8 6 6 6 9 8 8 4 8 6 6 

In [21]:
labels = list(range(len(val_dataset.classes)))
print("val classes:", val_dataset.classes)
print("val class_to_idx:", val_dataset.class_to_idx)

val classes: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
val class_to_idx: {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9}


In [22]:
from sklearn.metrics import confusion_matrix,classification_report

In [23]:
from sklearn.metrics import accuracy_score

print("VALIDATION RESULTS:")
model.eval()
print(actual.shape, predicted.shape)

matrix = confusion_matrix(actual, predicted, labels=labels)
report = classification_report(
    actual,
    predicted,
    labels=labels,
    target_names=val_dataset.classes,
    zero_division=0,
)
print("Accuracy:", accuracy_score(actual, predicted))
print("Confusion Matrix:\n", matrix)
print("Classification Report:\n", report)

VALIDATION RESULTS:
(5001,) (5001,)
Accuracy: 0.07558488302339532
Confusion Matrix:
 [[  0   0   2  11   6   6 159   0 274  32]
 [  7   0   2   5  63  18 211   0 184  78]
 [  1   0   2   9  15  15 174   0 270  30]
 [  0   0   2   8  13  24 234   0 194  30]
 [  2   0   1  17   6  23 185   0 231  26]
 [  3   0   1   7   5  11 185   0 190  44]
 [  4   0   5   4  15  17 162   0 240  32]
 [  4   0   2   9   8  49 244   0 169  29]
 [  1   0   1  10   6  18 223   0 166  62]
 [  2   0   2  15   4  24 203   0 232  23]]
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00       490
           1       0.00      0.00      0.00       568
           2       0.10      0.00      0.01       516
           3       0.08      0.02      0.03       505
           4       0.04      0.01      0.02       491
           5       0.05      0.02      0.03       446
           6       0.08      0.34      0.13       479
           7       0.00    

In [24]:
model.eval()
pred_batches = []
actual_batches = []
with torch.inference_mode():
    for x_train, y_train in train_loader:
        x_train, y_train = x_train.to(device), y_train.to(device)
        logits = model(x_train)
        y_pred = logits.argmax(dim=1).cpu().numpy()
        pred_batches.append(y_pred)
        actual_batches.append(y_train.cpu().numpy())

pred_train = np.concatenate(pred_batches, axis=0)
actual_train = np.concatenate(actual_batches, axis=0)

In [ ]:
print("TRAIN")
labels_train = list(range(len(train_dataset.classes)))
matrix_train = confusion_matrix(actual_train, pred_train, labels=labels_train)
report_train = classification_report(
    actual_train,
    pred_train,
    labels=labels_train,
    target_names=train_dataset.classes,
    zero_division=0,
)
print("Confusion Matrix:\n", matrix_train)
print("Classification Report:\n", report_train)

TRAIN
Confusion Matrix:
 [[ 0  0  0  0  0  0]
 [ 0  0  0  0 31  1]
 [ 0  0  0  0  2  1]
 [ 0  0  0  0  1  0]
 [ 0  0  0  0  0  0]
 [ 0  0  0  0  1  0]]
Classification Report:
               precision    recall  f1-score   support

           0       0.00      0.00      0.00      2962
           1       0.00      0.00      0.00      3371
           2       0.00      0.00      0.00      2979
           3       0.00      0.00      0.00      3066
           4       0.00      0.00      0.00      2921
           5       0.00      0.00      0.00      2711
           6       0.07      0.33      0.11      2959
           7       0.00      0.00      0.00      3133
           8       0.06      0.32      0.10      2926
           9       0.03      0.00      0.00      2975

    accuracy                           0.06     30003
   macro avg       0.02      0.07      0.02     30003
weighted avg       0.02      0.06      0.02     30003



/media/shaurya-singh/Shared-Data/EMBEDDED/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/media/shaurya-singh/Shared-Data/EMBEDDED/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/media/shaurya-singh/Shared-Data/EMBEDDED/.venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _

In [ ]:
MODEL_PATH = os.path.join(op, "model.pth")
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

Model saved. Listing contents of output directory again:


In [ ]:
np.save('/kaggle/output/train_loss.npy',train_loss)
np.save('/kaggle/output/test_loss.npy',test_loss)

## Factors for Comparing CNN and SNN Performance

When comparing your current CNN model with a future SNN model on the N-MNIST dataset, consider the following factors:

1.  **Classification Accuracy/Performance Metrics:**
    *   **Accuracy:** Overall percentage of correctly classified samples.
    *   **Precision, Recall, F1-Score:** Especially important for understanding performance per class, as N-MNIST can sometimes have class imbalances depending on the preprocessing.
    *   **Confusion Matrix:** Provides a detailed breakdown of correct and incorrect classifications for each class.

2.  **Computational Cost & Efficiency:**
    *   **Inference Latency:** How quickly each model can classify a single event stream. SNNs often offer lower latency, especially for event-driven data.
    *   **Energy Consumption:** A major advantage of SNNs is their potential for ultra-low power consumption due to sparse, event-driven computations. This can be estimated or measured if running on neuromorphic hardware or specialized simulators.
    *   **Number of Operations (FLOPs/MACs):** Compare the number of floating-point operations or multiply-accumulate operations during inference.

3.  **Memory Usage:**
    *   **Model Size (Parameters):** The total number of trainable parameters in each network.
    *   **Runtime Memory Footprint:** The memory required to run the model during inference.

4.  **Sparsity Exploitation:**
    *   How effectively each model handles the sparse, event-based nature of the N-MNIST data. SNNs are inherently designed for this, while CNNs process dense frames.
    *   **Event-per-classification efficiency:** For SNNs, how many events are needed to reach a decision.

5.  **Robustness and Generalization:**
    *   **Robustness to Noise:** How well each model performs with noisy or incomplete event streams.
    *   **Generalization to new data:** Performance on unseen data (already covered by your test set).

6.  **Training Complexity:**
    *   **Training Time:** How long it takes for each model to converge.
    *   **Hyperparameter Sensitivity:** How much tuning is required to achieve optimal performance.
    *   **Computational Resources for Training:** GPU/CPU time, memory usage during training.

7.  **Interpretability (Optional but good for research):**
    *   How easy it is to understand *why* a model made a certain decision. This can be more challenging for deep networks but sometimes SNNs offer insights into event processing.

By evaluating these factors, you can get a comprehensive understanding of the trade-offs and advantages of CNNs versus SNNs for your N-MNIST classification task.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
import time

def benchmark_cnn(model, device):
    model.eval()

    dummy_ip = torch.randn(1, 2, 30, 34, 34).to(device)

    torch.cuda.reset_peak_memory_stats(device)

    start_time = time.perf_counter()
    with torch.no_grad():
        for _ in range(100):
            _ = model(dummy_ip)
            if device.type == 'cuda':
                torch.cuda.synchronize()
    end_time = time.perf_counter()

    avg_latency_ms = ((end_time - start_time) / 100) * 1000

    # Measure Runtime Memory Footprint
    if device.type == 'cuda':
        peak_memory_bytes = torch.cuda.max_memory_allocated(device)
        peak_memory_mb = peak_memory_bytes / (1024 * 1024)
    else:
        peak_memory_mb = 0.0


    print("--- INFERENCE BENCHMARKS ---")
    print(f"Inference Latency: {avg_latency_ms:.4f} ms per event stream")
    print(f"Runtime Memory Footprint: {peak_memory_mb:.2f} MB")


In [ ]:
benchmark_cnn(model, device)

ValueError: Expected a cuda device, but got: cpu